# BERTScore cho kết quả ASR có sẵn

Notebook này chấm transcript đã sinh trong `output/benchmark_4_models/asr_benchmark/runs/<RUN_ID>/`. Nó không chạy lại ASR và chỉ ghi file mới vào thư mục `<RUN_ID>/bertscore/`; metrics gốc được giữ nguyên.

Mỗi run cần có `manifest.jsonl` cùng `predictions.jsonl` trong thư mục của từng model. PhoBERT được tải một lần vào cache local và dùng chung để chấm bốn model.


## 1. Cài dependency nếu cần

Notebook dùng `uv pip install --python <kernel-python>` để cài đúng vào môi trường Python đang chạy Jupyter. Nếu cần CUDA, hãy đảm bảo kernel đã dùng bản PyTorch phù hợp với driver; nếu các gói đã cài bằng `uv sync --extra asr`, có thể bỏ qua cell cài đặt.


In [6]:
# Cài dependency bằng uv vào đúng Python interpreter của Jupyter kernel.
import shutil
import subprocess
import sys

uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("Không tìm thấy uv. Cài uv, chọn kernel của môi trường dự án, rồi chạy lại cell này.")
subprocess.run(
    [uv, "pip", "install", "--python", sys.executable, "torch", "transformers==4.57.6", "bert-score==0.3.13", "pyvi==0.1.1"],
    check=True,
)


Using Python 3.12.14 environment at: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/.venv
Checked 4 packages in 9ms


CompletedProcess(args=['/home/fernir/.local/bin/uv', 'pip', 'install', '--python', '/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/.venv/bin/python', 'torch', 'transformers==4.57.6', 'bert-score==0.3.13', 'pyvi==0.1.1'], returncode=0)

## 2. Cấu hình run local

Đổi `RUN_ID` sang run cần đánh giá. Notebook liệt kê các run tìm được để bạn chọn chính xác. Đường dẫn kết quả mặc định là thư mục bạn đã chỉ định.


In [7]:
from __future__ import annotations

import csv
import gc
import json
import os
import sys
from pathlib import Path
from typing import Any

# Project root is discovered from the current working directory and its parents.
PROJECT_ROOT_OVERRIDE: Path | None = None
BENCHMARK_ROOT_OVERRIDE: Path | None = None
RUN_ID = "20261030T153536Z"  # Đổi thành ID của lần chạy ASR cần chấm.
MODEL_KEYS = [
    "phowhisper_medium",
    "phowhisper_large",
    "qwen3_asr_0_6b",
    "qwen3_asr_1_7b",
]
MODEL_KEYS_TO_SCORE = MODEL_KEYS.copy()  # Có thể chọn một vài model trong danh sách trên.

def find_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        root = PROJECT_ROOT_OVERRIDE.expanduser().resolve()
        if not (root / "src/asr_benchmark/configs/metrics.json").is_file():
            raise FileNotFoundError(f"Benchmark package config not found under {root}")
        return root
    for candidate in (Path.cwd(), *Path.cwd().resolve().parents):
        if (candidate / "src/asr_benchmark/configs/metrics.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot find project root. Set PROJECT_ROOT_OVERRIDE to the repository directory."
    )

PROJECT_ROOT = find_project_root()
PACKAGE_ROOT = PROJECT_ROOT / "src/asr_benchmark"
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
if BENCHMARK_ROOT_OVERRIDE is None:
    BENCHMARK_ROOT = PROJECT_ROOT / "output/benchmark_4_models/asr_benchmark"
else:
    BENCHMARK_ROOT = BENCHMARK_ROOT_OVERRIDE.expanduser().resolve()
RUNS_ROOT = BENCHMARK_ROOT / "runs"

if not RUNS_ROOT.is_dir():
    raise FileNotFoundError(f"Runs directory not found: {RUNS_ROOT}")
AVAILABLE_RUNS = sorted(path.name for path in RUNS_ROOT.iterdir() if path.is_dir())
print("Project root:", PROJECT_ROOT)
print("Benchmark root:", BENCHMARK_ROOT)
print("Available runs:", AVAILABLE_RUNS)
print("Selected run:", RUN_ID)


Project root: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge
Benchmark root: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/output/benchmark_4_models/asr_benchmark
Available runs: ['20261030T153536Z']
Selected run: 20261030T153536Z


## 3. Kiểm tra dữ liệu và dependency

Mỗi model được đối chiếu với manifest theo `sample_id`. Prediction thiếu được ghi nhận trong summary; ID lạ hoặc ID lặp sẽ làm notebook dừng để tránh chấm sai dữ liệu.


In [8]:
import importlib.util
import subprocess
import sys
import shutil
from importlib.metadata import PackageNotFoundError, version

# The setup cell above can be skipped: install missing or version-mismatched packages with uv into this kernel.
requirements = {
    "torch": ("torch", None),
    "transformers": ("transformers", "4.57.6"),
    "bert_score": ("bert-score", "0.3.13"),
    "pyvi": ("pyvi", "0.1.1"),
}
packages_to_install = []
for module, (package, expected_version) in requirements.items():
    installed = importlib.util.find_spec(module) is not None
    installed_version = None
    if installed and expected_version is not None:
        try:
            installed_version = version(package)
        except PackageNotFoundError:
            installed = False
    if not installed or (expected_version is not None and installed_version != expected_version):
        if module in sys.modules:
            raise RuntimeError(
                f"{module} is already loaded in this kernel but needs installation/update. "
                "Restart the kernel, then rerun this cell."
            )
        packages_to_install.append(package if expected_version is None else f"{package}=={expected_version}")

if packages_to_install:
    print("Installing dependencies into the current notebook kernel:", ", ".join(packages_to_install))
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("Không tìm thấy uv. Cài uv hoặc chạy `uv sync --extra asr` từ project root rồi chọn kernel đó.")
    subprocess.run(
        [uv, "pip", "install", "--python", sys.executable, *packages_to_install],
        check=True,
    )
    importlib.invalidate_caches()

missing = [label for module, (label, _) in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError(
        f"Could not install required packages: {', '.join(missing)}. "
        "Check network access and the uv output above."
    )

from asr_benchmark.infrastructure.evaluation.text_normalizer import normalize_text
from asr_benchmark.infrastructure.evaluation.semantic_metrics import PhoBERTBERTScore
from asr_benchmark.infrastructure.models.huggingface import download_snapshot
from asr_benchmark.infrastructure.storage.artifacts import write_json, write_jsonl
import torch

if RUN_ID not in AVAILABLE_RUNS:
    raise ValueError(f"RUN_ID {RUN_ID!r} not found. Choose one from AVAILABLE_RUNS.")
if not MODEL_KEYS_TO_SCORE or any(key not in MODEL_KEYS for key in MODEL_KEYS_TO_SCORE):
    raise ValueError("MODEL_KEYS_TO_SCORE must contain one or more supported model keys.")

RUN_DIR = RUNS_ROOT / RUN_ID
MANIFEST_PATH = RUN_DIR / "manifest.jsonl"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Run manifest not found: {MANIFEST_PATH}")

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Scoring device: {DEVICE}")
print("CUDA device:", torch.cuda.get_device_name(0) if DEVICE.startswith("cuda") else "CPU")


Scoring device: cpu
CUDA device: CPU


## 4. Đọc manifest, cấu hình và kiểm tra predictions


In [9]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

def unique_by_id(rows: list[dict[str, Any]], source: str) -> dict[str, dict[str, Any]]:
    indexed = {}
    duplicates = []
    for row in rows:
        sample_id = row.get("sample_id")
        if not sample_id:
            raise ValueError(f"Missing sample_id in {source}")
        if sample_id in indexed:
            duplicates.append(sample_id)
        indexed[sample_id] = row
    if duplicates:
        raise ValueError(f"Duplicate sample IDs in {source}: {duplicates[:10]}")
    return indexed

manifest_rows = read_jsonl(MANIFEST_PATH)
manifest = unique_by_id(manifest_rows, str(MANIFEST_PATH))
if not manifest:
    raise ValueError("Manifest is empty.")

metric_files = json.loads((PACKAGE_ROOT / "configs/metrics.json").read_text(encoding="utf-8"))
normalization_config = json.loads((PACKAGE_ROOT / "configs/normalization.json").read_text(encoding="utf-8"))
semantic_config = dict(metric_files["semantic_score"])
semantic_config["device"] = DEVICE
print(f"Manifest samples: {len(manifest)}")
print(f"Semantic model: {semantic_config['model_id']} (layers={semantic_config['num_layers']}, batch={semantic_config['batch_size']})")


Manifest samples: 1545
Semantic model: vinai/phobert-base (layers=12, batch=16)


## 5. Tải hoặc dùng cache PhoBERT

Model card PhoBERT yêu cầu word segmentation. Notebook gọi đúng lớp metric của benchmark, hiện dùng PyVi trước BERTScore. Cell dưới đọc HF_TOKEN từ file `.env` ở thư mục gốc dự án và truyền trực tiếp cho Hugging Face; token không được in ra output.


In [10]:
model_cache = PROJECT_ROOT / "artifacts/asr_benchmark/models" / f"semantic_{semantic_config['model_id'].replace('/', '--')}"
marker = model_cache / ".benchmark_model.json"
local_only = marker.is_file()
env_file = PROJECT_ROOT / ".env"
if not env_file.is_file():
    raise FileNotFoundError(f"HF_TOKEN file not found: {env_file}")
hf_token = None
for line in env_file.read_text(encoding="utf-8").splitlines():
    key, separator, value = line.partition("=")
    if separator and key.strip() == "HF_TOKEN":
        hf_token = value.strip()
        if len(hf_token) >= 2 and hf_token[0] in ("'", '"') and hf_token[-1] == hf_token[0]:
            hf_token = hf_token[1:-1]
        break
if not hf_token:
    raise ValueError(f"HF_TOKEN is missing or empty in {env_file}")
model_path, model_revision = download_snapshot(
    semantic_config["model_id"],
    semantic_config.get("revision"),
    model_cache,
    hf_token,
    local_files_only=local_only,
    allow_patterns=["*.json", "*.txt", "*.codes", "*.py", "*.safetensors", "pytorch_model.bin"],
)
semantic_config["revision"] = model_revision
print("PhoBERT snapshot:", model_path)
print("PhoBERT revision:", model_revision)


Fetching 8 files: 100%|██████████| 8/8 [02:34<00:00, 19.35s/it]

PhoBERT snapshot: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/artifacts/asr_benchmark/models/semantic_vinai--phobert-base
PhoBERT revision: 01daacda68afe13d83023d16ec647239e344a1e6


## 6. Tính điểm và ghi output riêng

Scores dùng macro average theo sample (trung bình điểm F1 từng câu), giống cách summary hiện tại tổng hợp semantic score. Điểm BERTScore giữ native scale, không đổi thành phần trăm.


In [11]:
scorer = PhoBERTBERTScore(str(model_path), semantic_config)
output_root = RUN_DIR / "bertscore"
output_root.mkdir(parents=True, exist_ok=True)
model_summaries = []

for model_key in MODEL_KEYS_TO_SCORE:
    prediction_path = RUN_DIR / model_key / "predictions.jsonl"
    if not prediction_path.is_file():
        raise FileNotFoundError(f"Predictions not found for {model_key}: {prediction_path}")
    predictions = unique_by_id(read_jsonl(prediction_path), str(prediction_path))
    unknown_ids = sorted(set(predictions) - set(manifest))
    if unknown_ids:
        raise ValueError(f"Unknown sample IDs in {model_key}: {unknown_ids[:10]}")
    missing_ids = [sample_id for sample_id in manifest if sample_id not in predictions]
    if missing_ids:
        print(f"{model_key}: {len(missing_ids)} manifest samples have no prediction; scoring available samples only.")

    sample_ids = [sample_id for sample_id in manifest if sample_id in predictions]
    references = [manifest[sample_id]["reference"] for sample_id in sample_ids]
    hypotheses = [predictions[sample_id]["text"] for sample_id in sample_ids]
    references_normalized = [normalize_text(text, normalization_config) for text in references]
    hypotheses_normalized = [normalize_text(text, normalization_config) for text in hypotheses]

    raw_scores = scorer.score_pairs(list(zip(references, hypotheses))) if semantic_config.get("evaluate_raw", True) else None
    normalized_scores = (
        scorer.score_pairs(list(zip(references_normalized, hypotheses_normalized)))
        if semantic_config.get("evaluate_normalized", True) else None
    )
    sample_rows = []
    for index, sample_id in enumerate(sample_ids):
        row = {
            "sample_id": sample_id,
            "reference": references[index],
            "prediction": hypotheses[index],
            "reference_normalized": references_normalized[index],
            "prediction_normalized": hypotheses_normalized[index],
        }
        if raw_scores is not None:
            row["raw_bertscore_f1"] = raw_scores[index]
        if normalized_scores is not None:
            row["normalized_bertscore_f1"] = normalized_scores[index]
        sample_rows.append(row)

    model_dir = output_root / model_key
    model_dir.mkdir(parents=True, exist_ok=True)
    write_jsonl(model_dir / "metrics.jsonl", sample_rows)
    summary = {
        "model": model_key,
        "expected_samples": len(manifest),
        "successful_samples": len(sample_rows),
        "missing_samples": len(missing_ids),
        "missing_sample_ids": missing_ids,
        "coverage": len(sample_rows) / len(manifest) if manifest else 1.0,
        "semantic_model": semantic_config["model_id"],
        "semantic_model_revision": model_revision,
        "semantic_model_parameter_count": scorer.model_parameter_count,
        "semantic_score_method": "BERTScore F1 (PhoBERT, unrescaled, no IDF)",
        "semantic_word_segmented": semantic_config.get("word_segment", True),
        "device": DEVICE,
        "batch_size": semantic_config["batch_size"],
    }
    if raw_scores is not None:
        summary["raw_bertscore_f1"] = sum(raw_scores) / len(raw_scores) if raw_scores else None
    if normalized_scores is not None:
        summary["normalized_bertscore_f1"] = sum(normalized_scores) / len(normalized_scores) if normalized_scores else None
    write_json(model_dir / "summary.json", summary)
    model_summaries.append(summary)
    print(f"{model_key}: raw={summary.get('raw_bertscore_f1')} normalized={summary.get('normalized_bertscore_f1')} coverage={summary['coverage']:.3f}")

write_json(output_root / "comparison.json", model_summaries)
if model_summaries:
    with (output_root / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(model_summaries[0]))
        writer.writeheader()
        writer.writerows(model_summaries)
    columns = ["model", "raw_bertscore_f1", "normalized_bertscore_f1", "coverage", "semantic_model_parameter_count"]
    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    for row in model_summaries:
        lines.append("| " + " | ".join(str(row.get(key, "")) for key in columns) + " |")
    (output_root / "comparison.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

print(f"\nBERTScore artifacts saved under: {output_root}")


/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/.venv/lib/python3.12/site-packages/pyvi/ViTokenizer.py:81: SyntaxWarning: invalid escape sequence '\.'
  specials = ["==>", "->", "\.\.\.", ">>",'\n']
/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/.venv/lib/python3.12/site-packages/pyvi/ViTokenizer.py:82: SyntaxWarning: invalid escape sequence '\d'
  digit = "\d+([\.,_]\d+)+"
/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/.venv/lib/python3.12/site-packages/pyvi/ViTokenizer.py:83: SyntaxWarning: invalid escape sequence '\.'
  email = "([a-zA-Z0-9_.+-]+@([a-zA-Z0-9-]+\.)+[a-zA-Z0-9-]+)"
/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/.venv/lib/python3.12/site-packages/pyvi/ViTokenizer.py:85: SyntaxWarning: invalid escape sequence '\w'
  web = "\w+://[^\s]+"
/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/.venv/lib/python3.12/site-packages/pyvi/ViTokenizer.py:90: SyntaxWarning: invalid escape se

phowhisper_medium: raw=0.7035531975690601 normalized=0.7610727525066018 coverage=1.000
phowhisper_large: raw=0.7230950263325836 normalized=0.7841230048716646 coverage=1.000
qwen3_asr_0_6b: raw=0.6787891546210039 normalized=0.7219907698700729 coverage=1.000
qwen3_asr_1_7b: raw=0.7367724952960092 normalized=0.7743990683054075 coverage=1.000

BERTScore artifacts saved under: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge/output/benchmark_4_models/asr_benchmark/runs/20261030T153536Z/bertscore


## 7. Dọn bộ nhớ GPU

Chạy sau khi đã lưu kết quả.


In [12]:
del scorer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("PhoBERT released from memory.")


PhoBERT released from memory.
